In [0]:
import re
from pyspark.sql import functions as F

base = "/Volumes/workspace/default/raw"

raw = (spark.readStream.format("cloudFiles")
       .option("cloudFiles.format", "csv")
       .option("header", True).option("multiLine", True).option("escape", '"')
       .option("cloudFiles.schemaLocation", f"{base}/_schema")
       .load(f"{base}/incoming/"))

# same snake_case rename as bronze, plus lineage columns
raw = raw.toDF(*[re.sub(r'[^a-z0-9]+', '_', c.strip().lower()).strip('_') for c in raw.columns])
raw = (raw.withColumn("source_file", F.col("_metadata.file_path"))
          .withColumn("ingest_ts", F.current_timestamp()))

(raw.writeStream
    .option("checkpointLocation", f"{base}/_checkpoint")
    .option("mergeSchema", "true")          # allows the new lineage columns
    .trigger(availableNow=True)             # load all new files, then stop
    .toTable("bronze_complaints")
    .awaitTermination())

display(spark.sql("SELECT source_file, COUNT(*) FROM bronze_complaints GROUP BY source_file"))